# Prepare `CVS-Act v1` For Hugging Face

This notebook prepares a Hugging Face-compatible dataset package for `CVS-Act v1` from the current local project artifacts.

What it does:
- exports the local `hf_cvs_act/` package from the current simple-action human and synthetic artifacts
- previews the generated dataset card and file layout
- optionally validates local loading with `datasets.load_dataset()`
- provides upload cells for pushing the prepared package to a Hugging Face dataset repo later

Important:
- this notebook does **not** upload anything unless you run the final upload cells
- the package currently exports configs `sages_trained_annotator` and `sages_synthetic`
- the config names encode source dataset first, while the actual split stays `test`
- no `surgeon_annotated` config is created unless actual surgeon-annotated data exists

Current naming convention:
- `sages_synthetic`: current split `test`, with room for future `train` / `validation`
- `sages_trained_annotator`: current split `test`
- future `sages_surgeon_annotated`: expected split `test`


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
EXPORT_SCRIPT = ROOT_DIR / 'scripts/export_hf_cvs_act.py'
PACKAGE_DIR = ROOT_DIR / 'hf_repos/cvs-act'
ENV_PATH = ROOT_DIR / '.env'

REPO_ID = 'BrachioLab/cvs-act'
PRIVATE = True
RELEASE_TAG = 'v1.0.0'
COMMIT_MESSAGE = 'Release CVS-Act synthetic train/test v1.0.0'
CREATE_PR = True


def load_env_file(path: Path) -> dict[str, str]:
    env = {}
    if not path.exists():
        return env
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        env[key] = value
    return env


env_from_file = load_env_file(ENV_PATH)
HF_TOKEN = os.environ.get('HF_TOKEN') or env_from_file.get('HF_TOKEN', '')

print('export script:', EXPORT_SCRIPT)
print('package dir:', PACKAGE_DIR)
print('env path:', ENV_PATH)
print('loaded HF_TOKEN from .env:', bool(env_from_file.get('HF_TOKEN')))
print('effective HF_TOKEN present:', bool(HF_TOKEN))


In [ ]:
assert EXPORT_SCRIPT.exists(), EXPORT_SCRIPT
PACKAGE_DIR.parent.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    [sys.executable, str(EXPORT_SCRIPT), '--out-dir', str(PACKAGE_DIR), '--validate'],
    check=True,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print('stderr:')
    print(result.stderr)


In [ ]:
assert PACKAGE_DIR.exists(), PACKAGE_DIR

all_paths = sorted(
    path.relative_to(PACKAGE_DIR)
    for path in PACKAGE_DIR.rglob('*')
    if path.is_file()
)
all_paths


In [ ]:
readme_path = PACKAGE_DIR / 'README.md'
print(readme_path.read_text())


In [ ]:
taxonomy_path = PACKAGE_DIR / 'taxonomy/action_taxonomy.json'
taxonomy = json.loads(taxonomy_path.read_text())
taxonomy


## Optional Upload

Run the cells below only when you are ready to create or update the Hugging Face dataset repo.

On the Hugging Face web side you need:
1. a Hugging Face account
2. a write token from `Settings -> Access Tokens`
3. optionally a pre-created dataset repo, though the API cell below can create it automatically

If you want the repo private first, leave `PRIVATE = True` in the config cell above.


In [ ]:
try:
    from huggingface_hub import HfApi
except Exception as exc:
    raise RuntimeError(
        'Missing huggingface_hub. Install it with: pip install huggingface_hub'
    ) from exc

assert HF_TOKEN, 'Set HF_TOKEN in your environment or assign it in the config cell.'

api = HfApi(token=HF_TOKEN)
whoami = api.whoami()
print('authenticated as:', whoami.get('name') or whoami)


In [ ]:
repo_url = api.create_repo(
    repo_id=REPO_ID,
    repo_type='dataset',
    private=PRIVATE,
    exist_ok=True,
)
print('repo:', repo_url)


In [ ]:
result = api.upload_folder(
    repo_id=REPO_ID,
    repo_type='dataset',
    folder_path=str(PACKAGE_DIR),
    path_in_repo='.',
    commit_message=COMMIT_MESSAGE,
    create_pr=CREATE_PR,
)
print('commit:', result.commit_url)
print('commit hash:', result.oid)
print('pr:', getattr(result, 'pr_url', None))
result


In [ ]:
if CREATE_PR:
    print(f'Upload was created as a PR. Merge it first, then create tag {RELEASE_TAG} on main.')
else:
    api.create_tag(
        repo_id=REPO_ID,
        repo_type='dataset',
        tag=RELEASE_TAG,
        revision=result.oid,
        tag_message='CVS-Act v1.0.0 synthetic train/test release',
        exist_ok=False,
    )
    print('tagged:', RELEASE_TAG)
